# Run a Single Coupled Example - Dynamic Wflow -> SFINCS

Run one coupled Wflow-SFINCS event: Wflow starts from the prepared native instate, generates SFINCS inflow hydrographs, then SFINCS consumes that dynamic handoff.

Stage Contract
- Requires: built coupled Wflow/SFINCS bases, Event Catalog, warmup forcing/states from `b_prepare_wflow_dynamic_handoff.ipynb`, and reviewed source artifacts.
- Produces: dynamic Wflow `sfincs_discharge.nc`, dynamic handoff QA/acceptance, one staged SFINCS scenario folder, forcing QA plots, optional SFINCS outputs, and post-run diagnostics.
- Next: review diagnostics or batch scenarios in `../05_create_scenarios.ipynb`.


In [ ]:
import json
import os
import warnings
from pathlib import Path

os.environ.pop("DEBUG", None)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/flood-rm-matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import HTML, Video, display

# standard paths and readiness tables.
from wflow_runs.notebook import load_runtime
# event selection, SFINCS staging, and handoff readiness.
from sfincs_runs.scenarios import (
    plan_example,
    stage_inland_coupled_example_forcing,
)
# stage one SFINCS run from the Event Catalog.
from sfincs_runs.scenarios.event_forcing import run_model
# review forcing, flood animations, and post-run checks.
from sfincs_runs.diagnostics import (
    plot_forcing,
    plot_diagnostics,
    plot_animation,
)
# meteo forcing, discharge handoff, and acceptance gates.
from wflow_runs import ensure_dynamic_handoff

runtime = load_runtime(Path("../..").resolve(), wflow_domain_review_required=False)
location_root = runtime.location_root
repo_root = runtime.repo_root
config = runtime.config
paths = runtime.paths
events_root = runtime.events_root


## 1 - Select a catalog event

`plan_example` checks the static/base-model contract and selects the catalog event used by both Wflow and SFINCS.


In [ ]:
EVENT_ID = None  # None -> highest-RP catalog event; or e.g. "design_0398"
CATALOG_PATH = "data/event_catalog/catalog/probability_catalog.csv"

example_plan = plan_example(
    config,
    {"location_root": location_root},
    catalog_path=CATALOG_PATH,
    event_id=EVENT_ID,
)

plan_summary = pd.Series({
    "status": example_plan.status,
    "event_id": example_plan.event_id,
    "event_reference_time": example_plan.event_reference_time,
    "catalog_path": example_plan.catalog_path,
    "handoff_path": example_plan.handoff_path,
    "wflow_event_dir": example_plan.wflow_event_dir,
    "wflow_discharge_forcing": example_plan.wflow_discharge_forcing,
    "sfincs_scenario_dir": example_plan.sfincs_scenario_dir,
    "sfincs_dry_run_command": example_plan.sfincs_dry_run_command,
}, name="inland_coupled_example_plan")
display(plan_summary)

if example_plan.issues:
    print("Blocking issues (resolve upstream, then rerun):")
    for issue in example_plan.issues:
        print("  -", issue)
if example_plan.status != "ready":
    raise RuntimeError("Coupled example inputs are not ready; resolve the blocking issues above.")

event_id = example_plan.event_id
discharge_nc = events_root / event_id / "sfincs_discharge.nc"
event_precip_nc = events_root / event_id / "precip.nc"
acceptance_json = events_root / event_id / "sfincs_discharge.dynamic_handoff.json"


## Rerun Control


In [ ]:
rerun = True


## 2 - Run dynamic Wflow handoff

Run Wflow first from the prepared native `instate/instates.nc`. The accepted `sfincs_discharge.nc` produced here is the upstream boundary forcing that SFINCS consumes at its native `src` inflow points.


In [ ]:
run_dynamic_wflow_handoff = True
handoff_run = ensure_dynamic_handoff(
    config,
    location_root,
    event_id,
    catalog_path=CATALOG_PATH,
    rerun=rerun,
    run=run_dynamic_wflow_handoff,
)

display(handoff_run.summary)
if handoff_run.meteo_report is not None:
    display(handoff_run.meteo_report)
if handoff_run.handoff_report is not None:
    display(handoff_run.handoff_report)

handoff_acceptance = handoff_run.acceptance
display(handoff_acceptance)


## 3 - Configure SFINCS run


In [ ]:
config["scenario_run"]["threads"] = 8
run_sfincs_solver = True

display(pd.Series({
    "dynamic_wflow_handoff": "accepted",
    "sfincs_solver": "run" if run_sfincs_solver else "stage forcing only",
    "sfincs_threads": config["scenario_run"]["threads"],
}, name="interactive_model_run"))

## 4 - Stage SFINCS and apply rainfall + dynamic Wflow discharge


In [ ]:
forcing_stage = stage_inland_coupled_example_forcing(
    config,
    {"location_root": location_root},
    example_plan=example_plan,
    event_id=event_id,
    force=rerun,
)

scenario_report = forcing_stage.scenario_report
sfincs_report = forcing_stage.sfincs_report
sim = forcing_stage.sim

display(scenario_report[["event_id", "sfincs_domain_id", "scenario_status", "run_root", "wflow_discharge_forcing"]])
display(sfincs_report[["event_id", "sfincs_domain_id", "status", "n_src", "run_root", "wflow_discharge_forcing"]])
for _, row in sfincs_report.iterrows():
    print(f"{row['sfincs_domain_id']}: staged forcing written; run pre-run QA before launching SFINCS.")


## 5 - Wflow handoff output

Review the dynamic Wflow response that is handed to SFINCS: source peak discharge, staged rainfall total when direct rainfall is enabled, and handoff hydrographs.


In [ ]:
from wflow_runs import plot_wflow_event_handoff

precip_for_plot = event_precip_nc if event_precip_nc.exists() else None
fig, axes = plot_wflow_event_handoff(
    discharge_nc,
    precipitation_nc=precip_for_plot,
    event_label=event_id,
)


## 6 - Pre-run forcing QA


In [ ]:
for domain_id, s in sim.items():
    plot_forcing(
        forcing_manifest=s["run_dir"] / "forcing_manifest.json",
        event_id=event_id,
        event_label=f"{event_id} · {domain_id}",
    )

if not sim:
    print("No example runs — nothing to QA.")


## 7 - Run SFINCS


In [ ]:
if run_sfincs_solver:
    sfincs_rows = []
    for domain_id, s in sim.items():
        run_dir = s["run_dir"]
        result = run_model(run_dir, config=config)
        s["result"] = result
        manifest_path = run_dir / "forcing_manifest.json"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        manifest["sfincs_run_executed"] = True
        manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        sfincs_rows.append({
            "event_id": event_id,
            "sfincs_domain_id": domain_id,
            "status": "completed",
            "n_src": int(s["dis"].sizes.get("index", 0)),
            "run_root": str(run_dir),
            "map_written": result.map_path.exists(),
        })
    sfincs_run_report = pd.DataFrame(sfincs_rows)
    display(sfincs_run_report[["event_id", "sfincs_domain_id", "status", "n_src", "run_root", "map_written"]])
else:
    print("SFINCS solver skipped; staged forcing and QA are available for review.")


## 8 - Flood + discharge animation (mp4)


In [ ]:
for domain_id, s in sim.items():
    discharge_df = s["dis"].transpose("time", "index").to_pandas()
    out_mp4 = plot_animation(
        run_root=s["run_dir"],
        out_dir=s["run_dir"] / "figures",
        event_id=event_id,
        event_label=f"{event_id} · {domain_id}",
        discharge=discharge_df,
        t_start=s["t_start"],
    )
    display(HTML(f"<h4>{event_id} · {domain_id} — flood + discharge animation</h4>"))
    display(Video(str(out_mp4), embed=True))

if not sim:
    print("No example runs — nothing to animate.")
